In [46]:
import json
import pandas as pd
from pathlib import Path
import requests

COMMUNITY_FILE = Path("../data/results/community_assignments.csv")
TRIPLES_FILE   = Path("../data/results/community_triples.json")

df = pd.read_csv(COMMUNITY_FILE)
with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Communities: {df['community_id'].nunique()}")

Communities: 94


In [47]:
def build_summary(comm_df):
    label = comm_df["Label"].value_counts().idxmax()
    tactic = comm_df["attck_tactic"].mode().iloc[0]
    ports = comm_df["Destination Port"].value_counts().head(3)
    port_str = ", ".join([str(p) for p in ports.index])

    return f"""
Dominant IDS label: {label}
Ground-truth tactic for evaluation: {tactic}
Top destination ports: {port_str}
"""

In [48]:
def get_graph_features(community_id):
    triples = community_triples.get(str(community_id), [])

    features = []
    for t in triples:
        features.append(f"{t['relation']} → {t['object']}")

    return "\n".join(features[:6])  # keep it short

In [49]:
def build_prompt(summary, graph_features):
    return f"""
You are a cybersecurity analyst classifying SIEM alert communities.

Choose exactly one output from this list:
- T1046
- T1110.001
- T1498.001
- T1499.001
- T1071.001
- T1105
- T1059.007
- T1190
- BENIGN

You must follow these priority rules in order:

1. If the dominant IDS label is BENIGN and there is no strong attack evidence, output BENIGN.
2. If the dominant IDS label contains FTP Patator or SSH Patator, output T1110.001.
3. If the dominant IDS label contains PortScan, output T1046.
4. If the dominant IDS label contains DDoS, output T1498.001.
5. If the dominant IDS label contains DoS, output T1499.001.
6. If the dominant IDS label contains Bot, output T1071.001.
7. If the dominant IDS label contains Web Attack Sql Injection or Heartbleed, output T1190.
8. If the dominant IDS label contains Web Attack XSS, output T1059.007.
9. If none of the above clearly apply, output BENIGN.

Use the extracted relations only as supporting evidence, not as the main source of truth.

COMMUNITY SUMMARY:
{summary}

EXTRACTED RELATIONS:
{graph_features}

Return ONLY JSON:
{{
  "technique_id": "...",
  "confidence": "high/medium/low",
  "reason": "one short sentence"
}}
"""

In [50]:
def call_ollama(prompt):
    payload = {
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 200
        }
    }

    response = requests.post("http://localhost:11434/api/generate",
                             json=payload, timeout=300)
    return response.json().get("response", "")

In [51]:
results = []

sample_comms = df["community_id"].unique()[:10]

for cid in sample_comms:
    comm_df = df[df["community_id"] == cid]

    summary = build_summary(comm_df)
    graph_features = get_graph_features(cid)

    prompt = build_prompt(summary, graph_features)

    output = call_ollama(prompt)

    print(f"\nCommunity {cid}")
    print(output)


Community 0
{
  "technique_id": "T1498.001",
  "confidence": "high",
  "reason": "The dominant IDS label is DDoS, which matches T1498.001."
}

Community 1
{
  "technique_id": "T1046",
  "confidence": "high",
  "reason": "The dominant IDS label is PortScan, which corresponds to T1046."
}

Community 2
{
  "technique_id": "T1071.001",
  "confidence": "high",
  "reason": "The dominant IDS label is 'Bot' and the extracted relations indicate a long duration and high packet count, which aligns with Command And Control communication."
}

Community 3
{
  "technique_id": "T1110.001",
  "confidence": "high",
  "reason": "The dominant IDS label is FTP Patator, which corresponds to T1110.001."
}

Community 4
{
  "technique_id": "BENIGN",
  "confidence": "high",
  "reason": "The dominant IDS label is BENIGN and there is no strong attack evidence."
}

Community 5
{
  "technique_id": "T1110.001",
  "confidence": "high",
  "reason": "Dominant IDS label contains SSH Patator, which corresponds to T1110.

In [52]:
results = []

sample_comms = df["community_id"].unique()[:50]

for cid in sample_comms:
    comm_df = df[df["community_id"] == cid]

    summary = build_summary(comm_df)
    graph_features = get_graph_features(cid)

    prompt = build_prompt(summary, graph_features)
    output = call_ollama(prompt)

    try:
        pred = json.loads(output)
        pred_id = pred.get("technique_id", "")
        confidence = pred.get("confidence", "")
        reason = pred.get("reason", "")
    except:
        pred_id = ""
        confidence = ""
        reason = ""

    gt_id = comm_df["attck_technique_id"].mode().iloc[0]

    results.append({
        "community_id": cid,
        "ground_truth": gt_id,
        "predicted": pred_id,
        "correct": pred_id == gt_id,
        "confidence": confidence,
        "reason": reason
    })

results_df = pd.DataFrame(results)
results_df.head()

,community_id,ground_truth,predicted,correct,confidence,reason
0,0,T1498.001,T1498.001,True,high,"The dominant IDS label is DDoS, which matches ..."
1,1,T1046,T1046,True,high,"The dominant IDS label is PortScan, which corr..."
2,2,T1071.001,T1071.001,True,high,The dominant IDS label is 'Bot' and the extrac...
3,3,T1110.001,T1110.001,True,high,"The dominant IDS label is FTP Patator, which c..."
4,4,BENIGN,BENIGN,True,high,The dominant IDS label is BENIGN and there is ...


In [53]:
print("Accuracy:", results_df["correct"].mean())
print(results_df["correct"].value_counts())

Accuracy: 0.96
correct
True     48
False     2
Name: count, dtype: int64


In [54]:
RESULTS_FILE = Path("../data/results/stage5_classification_results.csv")
results_df.to_csv(RESULTS_FILE, index=False)
print(f"Saved to {RESULTS_FILE}")

Saved to ../data/results/stage5_classification_results.csv
